# CircuitKit — 2.5-Minute Demo

A single walk through the full pipeline on **GPT-2 / IOI**: discover a circuit, visualize it, evaluate its faithfulness, use it to prune the model, and run the same path on your own data.

> **For recording:** run this notebook top to bottom once so every output is cached, then screen-record a scroll-through with voiceover. Nothing needs to run live on camera.

## 1. Load a model and a task

In [ ]:
from circuitkit import Pipeline
import circuitkit as ck

# GPT-2 + the classic Indirect-Object-Identification task. Runs on CPU.
pipe = Pipeline("gpt2", task="ioi", precision="float32",
                output_dir="./demo_out")
print(pipe)

## 2. Discover a circuit — one call

In [ ]:
pipe.discover(algorithm="eap-ig", n_examples=256, batch_size=2,
              ig_steps=3, sparsity=0.3)

print("Top circuit components:")
for name, score in pipe._circuit.top_nodes(10).items():
    print(f"  {name:>10s}  {score:.4f}")

## 3. Visualize the circuit

The interactive graph should surface the **Name-Mover** heads that define the IOI circuit.

In [ ]:
from IPython.display import HTML, display

html = pipe.visualize(mode="graph", output="./demo_out/circuit_graph.html")
display(HTML(html))

## 4. Evaluate faithfulness — six pillars, one summary

In [ ]:
pipe.evaluate(
    pillars=["patching", "ablation", "stability",
             "robustness", "baselines"],
    n_examples=128, n_stability_runs=3,
)
pipe.summary()   # formatted per-pillar table

## 5. Intervene: prune, export, benchmark

The same artifact drives a downstream intervention with no conversion. We compare the circuit-guided cut against a magnitude baseline at the same sparsity.

In [ ]:
# Circuit-guided structural pruning + export to a standard HF checkpoint
pipe.prune(sparsity=0.3, scope="both")
pipe.export("./demo_out/pruned_circuit")

# Benchmark the pruned checkpoint (limit kept small for a fast demo run)
scores = ck.benchmark("./demo_out/pruned_circuit", tasks=["boolq"],
                      fewshot=0, limit=100)
print("Circuit-guided pruned accuracy:", scores["boolq"])

## 6. The differentiator: discover on *your own* data

No canonical benchmark required. You supply the data as a CSV (or JSON) and write clean/corrupt prompt templates whose placeholders are your column names. CircuitKit fills the templates row by row and pairs your clean and corrupt prompts, then handles tokenization and length alignment. It does not synthesize the data itself: every column referenced in the template must already exist in your file.

In [ ]:
CLEAN_PROMPT   = ("System: {system_jailbreak}\n"
                  "User: {benign_req}\n"
                  "Please answer with only 'Yes' or 'No'.\nAssistant:")
CORRUPT_PROMPT = ("System: {system_jailbreak}\n"
                  "User: {harmful_req}\n"
                  "Please answer with only 'Yes' or 'No'.\nAssistant:")

custom = Pipeline.from_custom_data(
    "gpt2",
    data_path="src/circuitkit/data/task_data/tasks/binary_align/jailbreak_binary.csv",
    clean_prompt=CLEAN_PROMPT,   corrupt_prompt=CORRUPT_PROMPT,
    clean_answer="{clean_ans}",  corrupt_answer="{corrupt_ans}",
    task_name="jailbreak_demo", output_dir="./demo_out/custom",
)
custom.discover(algorithm="eap-ig", level="node", sparsity=0.3,
                n_examples=128, batch_size=8, ig_steps=5)

print("Circuit discovered on custom data:")
for name, score in custom._circuit.top_nodes(8).items():
    print(f"  {name:>10s}  {score:.4f}")

## One pipeline, three interfaces

Everything above is also a flat one-shot API and a YAML-driven CLI over the **same** `CircuitScores` artifact:

```python
# flat API
circuit = ck.discover(model, task="ioi", algorithm="eap-ig")
```
```bash
# CLI
circuitkit discover -m gpt2 -a eap-ig -t ioi -s 0.3 -o circuit.pt
circuitkit run pipeline.yaml
```

**Source-available (LSAL v1.1).** https://github.com/Lexsi-Labs/circuitkit